Phase 0 : Mise en route

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

Phase A : Prédire les prix immobiliers (régression)

In [ ]:

def charger_immobilier():
    """Charge California Housing, renvoie X, y.
    Doit afficher : nombre de lignes, nombre de variables, et l'unité de la cible.
    """
    # TODO : data = fetch_california_housing()
    data = fetch_california_housing()
    # TODO : afficher data.data.shape et data.feature_names
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = data.target
    print(f"California Housing : {X.shape} variables, cible = prix médian en centaines de milliers de $")
    # TODO : renvoyer X, y
    return X, y

In [9]:
def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    """Entraîne, prédit, renvoie un dict {r2, mae, rmse}.
    Doit renvoyer les 3 métriques de régression vues en section 2.
    """
    # TODO : fit, predict
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    # TODO : calculer r2_score, mean_absolute_error, root_mean_squared_error
    r2 = sklearn.metrics.r2_score(y_test, y_pred)
    mae = sklearn.metrics.mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(sklearn.metrics.mean_squared_error(y_test, y_pred))
    # TODO : renvoyer {"r2": ..., "mae": ..., "rmse": ...}
    return {"r2": r2, "mae": mae, "rmse": rmse}

In [20]:
X, y = charger_immobilier()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = make_pipeline(StandardScaler(), LinearRegression())
rf = make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=100, random_state=42))

res_lr = evaluer_regression(lr, X_train, X_test, y_train, y_test)
res_rf = evaluer_regression(rf, X_train, X_test, y_train, y_test)
print(f"LinearRegression : R2={res_lr['r2']:.2f} MAE={res_lr['mae']:.2f} RMSE={res_lr['rmse']:.2f}")
print(f"RandomForest     : R2={res_rf['r2']:.2f} MAE={res_rf['mae']:.2f} RMSE={res_rf['rmse']:.2f}")

California Housing : (20640, 8) variables, cible = prix médian en centaines de milliers de $
LinearRegression : R2=0.58 MAE=0.53 RMSE=0.75
RandomForest     : R2=0.81 MAE=0.33 RMSE=0.51


### Cas limite : entraînement sur 100 lignes

Oui, le **R² peut fortement baisser**. Avec seulement 100 lignes, le modèle a moins d'exemples pour apprendre et ses prédictions sont moins fiables.

**Pourquoi ?**
Un modèle a besoin de beaucoup de données pour comprendre les relations entre les variables et faire de bonnes prédictions.

---

### Cas adversarial : revenu médian = 0 et 9000 habitants

Le modèle peut renvoyer une valeur peu réaliste, car ces données sont très différentes de celles utilisées pour l'entraînement.

**En production :**
Il faut vérifier que les données d'entrée sont valides et refuser ou signaler les valeurs hors plage avant de faire la prédiction.


Phase B : Segmenter les clients d'AirBnB (non supervisé)

In [ ]:
def charger_airbnb(url_csv):
    df = pd.read_csv(url_csv)

    colonnes_utiles = ['price', 'minimum_nights', 'number_of_reviews', 'availability_365']
    colonnes_utiles = [col for col in colonnes_utiles if col in df.columns]
    X = df[colonnes_utiles].copy()

    for col in X.columns:
        if X[col].dtype == object:
            X[col] = pd.to_numeric(
                X[col].astype(str).str.replace(r'[^\d.]', '', regex=True),
                errors='coerce'
            )

    X = X.loc[:, X.isna().mean() < 0.8]
    print("Colonnes retenues après filtre NaN:", X.columns.tolist())

    X = X.dropna()
    print(f"Listings chargés : {X.shape[0]} lignes, {X.shape[1]} colonnes numériques retenues")
    return X

In [46]:
def choisir_k(X_scaled, k_range=range(2, 9)):
    """Pour chaque k, renvoie inertie et silhouette.
    Doit afficher un tableau permettant de repérer le coude / le meilleur k.
    """
    resultats = []
    # TODO : pour chaque k, fit KMeans(n_init=10), récupérer inertia_ et silhouette_score
    for k in k_range:
        kmeans = sklearn.cluster.KMeans(n_clusters=k, n_init=10, random_state=42)
        kmeans.fit(X_scaled)

        inertie = kmeans.inertia_
        silhouette = sklearn.metrics.silhouette_score(X_scaled, kmeans.labels_)
        resultats.append((k, inertie, silhouette))
        # TODO : afficher k, inertie, silhouette
        print(f"k={k} : inertie={inertie:.2f}, silhouette={silhouette:.2f}")
    
    k_retenu = max(resultats, key=lambda x: x[2])[0]
    print(f"nSegment retenu : {k_retenu}")        

In [47]:
url_csv = "data/listings.csv"
df_airbnb = charger_airbnb(url_csv)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_airbnb)

choisir_k(X_scaled)


Colonnes retenues après filtre NaN: ['minimum_nights', 'number_of_reviews', 'availability_365']
Listings chargés : 453 lignes, 3 colonnes numériques retenues
k=2 : inertie=988.69, silhouette=0.48
k=3 : inertie=699.52, silhouette=0.48
k=4 : inertie=429.99, silhouette=0.52
k=5 : inertie=319.44, silhouette=0.54
k=6 : inertie=248.85, silhouette=0.54
k=7 : inertie=210.76, silhouette=0.46
k=8 : inertie=179.69, silhouette=0.48
nSegment retenu : 5


### Cas normal

5 segments ont été identifiés. Chaque segment correspond à un groupe d'annonces ayant des caractéristiques similaires.

### Cas limite

Sans standardisation, les clusters ont moins de sens car une variable avec de grandes valeurs peut dominer les autres et influencer le résultat.

### Cas adversarial

Une annonce à 100 000 € déplace les centres des clusters et fausse le regroupement. C'est pourquoi le nettoyage des valeurs aberrantes est indispensable avant d'appliquer KMeans.
